In [1]:
# ================================================================
# ECHOLOCK: AI DATA INTEGRITY INTELLIGENCE ENGINE
# Complete One-Cell Project | Data Science + ML + Visualizations
# ================================================================
# FEATURES:
# 1. Synthetic dataset generation
# 2. Missing-value and duplicate detection
# 3. Invalid-value and business-rule validation
# 4. Conflicting-record detection
# 5. Isolation Forest anomaly detection
# 6. Data drift detection
# 7. Explainable anomaly indicators
# 8. Data reliability scoring
# 9. Interactive Plotly visualizations
# 10. Exportable CSV and JSON reports
#
# IMPORTANT:
# This is a demonstration using synthetic data.
# An anomaly is a candidate for investigation, not proof of fraud.
# ================================================================

import sys
import subprocess
import importlib.util
import warnings
warnings.filterwarnings("ignore")

# Install missing dependencies automatically
for package, module in [
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("scikit-learn", "sklearn"),
    ("plotly", "plotly"),
]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", package]
        )

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from IPython.display import display

from datetime import datetime, timedelta
import json
import os

# -----------------------------
# 1. PROJECT CONFIGURATION
# -----------------------------

SEED = 42
N_RECORDS = 1500
OUTPUT_DIR = "echolock_outputs"

np.random.seed(SEED)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("=" * 75)
print(" ECHOLOCK | AI DATA INTEGRITY INTELLIGENCE ENGINE")
print("=" * 75)
print("Generating demonstration data and initializing analysis...\n")

# -----------------------------
# 2. GENERATE SYNTHETIC DATA
# -----------------------------

departments = ["Retail", "Healthcare", "Finance", "Technology", "Operations"]
regions = ["North", "South", "East", "West", "Central"]
statuses = ["Approved", "Pending", "Rejected", "Under Review"]

n = N_RECORDS

dates = pd.date_range(
    end=pd.Timestamp.today().normalize(),
    periods=n,
    freq="h"
)

df = pd.DataFrame({
    "record_id": [f"REC{i:05d}" for i in range(1, n + 1)],
    "customer_id": [f"CUST{np.random.randint(1000, 1300)}" for _ in range(n)],
    "timestamp": dates,
    "department": np.random.choice(
        departments, n, p=[0.25, 0.18, 0.22, 0.20, 0.15]
    ),
    "region": np.random.choice(regions, n),
    "age": np.random.normal(38, 13, n).round().astype(float),
    "transaction_amount": np.random.lognormal(5.0, 0.75, n).round(2),
    "transaction_count": np.random.poisson(3, n).astype(float),
    "processing_time_seconds": np.random.gamma(3, 2, n).round(2),
    "status": np.random.choice(
        statuses, n, p=[0.65, 0.18, 0.10, 0.07]
    ),
})

df["age"] = df["age"].clip(18, 85)

# Add a realistic temporal distribution shift to recent records.
# This is deliberately injected so drift detection can be demonstrated.
recent_start = int(n * 0.75)
recent_idx = df.index[recent_start:]

df.loc[recent_idx, "transaction_amount"] *= 1.65
df.loc[recent_idx, "processing_time_seconds"] *= 1.35

# Preserve a clean reference copy for demonstrating injected defects.
df["declared_amount"] = df["transaction_amount"].copy()

# -----------------------------
# 3. INJECT DEMONSTRATION ISSUES
# -----------------------------

# Missing values
missing_amount_idx = np.random.choice(df.index, 35, replace=False)
df.loc[missing_amount_idx, "transaction_amount"] = np.nan

missing_age_idx = np.random.choice(df.index, 22, replace=False)
df.loc[missing_age_idx, "age"] = np.nan

missing_region_idx = np.random.choice(df.index, 18, replace=False)
df.loc[missing_region_idx, "region"] = np.nan

# Invalid values
invalid_age_idx = np.random.choice(df.index, 8, replace=False)
df.loc[invalid_age_idx, "age"] = np.random.choice(
    [-5, 0, 130, 160], len(invalid_age_idx)
)

invalid_amount_idx = np.random.choice(df.index, 12, replace=False)
df.loc[invalid_amount_idx, "transaction_amount"] = -np.random.uniform(
    1, 500, len(invalid_amount_idx)
).round(2)

invalid_count_idx = np.random.choice(df.index, 8, replace=False)
df.loc[invalid_count_idx, "transaction_count"] = -1

invalid_status_idx = np.random.choice(df.index, 10, replace=False)
df.loc[invalid_status_idx, "status"] = "UNKNOWN_STATUS"

# Exact duplicate records
exact_duplicate_source = df.sample(12, random_state=SEED)
df = pd.concat([df, exact_duplicate_source], ignore_index=True)

# Conflicting records: same record ID, but a different amount
conflict_source = df.sample(12, random_state=SEED + 10).copy()
conflict_source["transaction_amount"] = (
    conflict_source["transaction_amount"] * 2.4
)
conflict_source["declared_amount"] = conflict_source["declared_amount"]
df = pd.concat([df, conflict_source], ignore_index=True)

# Duplicate indices are no longer relevant after concatenation.
df = df.reset_index(drop=True)

# Save the synthetic raw dataset.
df.to_csv(
    os.path.join(OUTPUT_DIR, "synthetic_raw_dataset.csv"),
    index=False
)

print(f"Dataset generated: {len(df):,} rows")
print(f"Columns: {len(df.columns)}")
print(f"Output directory: {OUTPUT_DIR}/")

# -----------------------------
# 4. INITIAL DATASET AUDIT
# -----------------------------

print("\n" + "=" * 75)
print("STEP 1 | DATASET OVERVIEW")
print("=" * 75)

overview = pd.DataFrame({
    "Metric": [
        "Total records",
        "Total columns",
        "Exact duplicate rows",
        "Rows containing missing values",
        "Total missing cells",
        "Unique record IDs",
        "Unique customers",
        "Date range start",
        "Date range end"
    ],
    "Value": [
        len(df),
        len(df.columns),
        int(df.duplicated().sum()),
        int(df.isna().any(axis=1).sum()),
        int(df.isna().sum().sum()),
        int(df["record_id"].nunique()),
        int(df["customer_id"].nunique()),
        str(df["timestamp"].min()),
        str(df["timestamp"].max())
    ]
})

display(overview)

# -----------------------------
# 5. MISSING-VALUE ANALYSIS
# -----------------------------

print("\n" + "=" * 75)
print("STEP 2 | MISSING-VALUE ANALYSIS")
print("=" * 75)

missing_report = pd.DataFrame({
    "column": df.columns,
    "missing_count": df.isna().sum().values,
    "missing_percentage": (
        df.isna().mean().values * 100
    ).round(2)
}).sort_values("missing_count", ascending=False)

display(missing_report[missing_report["missing_count"] > 0])

# -----------------------------
# 6. DUPLICATE & CONFLICT ANALYSIS
# -----------------------------

print("\n" + "=" * 75)
print("STEP 3 | DUPLICATE AND CONFLICT ANALYSIS")
print("=" * 75)

exact_duplicate_mask = df.duplicated(keep=False)

# An ID is conflicting if its non-ID data varies across records.
compare_cols = [
    c for c in df.columns
    if c not in ["record_id"]
]

variation_counts = df.groupby("record_id", dropna=False)[
    compare_cols
].nunique(dropna=False)

conflicting_ids = variation_counts.index[
    (variation_counts > 1).any(axis=1)
]

conflict_mask = df["record_id"].isin(conflicting_ids)

print("Rows participating in exact duplicate groups:",
      int(exact_duplicate_mask.sum()))
print("Unique record IDs with conflicting values:",
      int(len(conflicting_ids)))
print("Rows associated with conflicting IDs:",
      int(conflict_mask.sum()))

conflict_report = df.loc[
    conflict_mask,
    ["record_id", "customer_id", "timestamp",
     "transaction_amount", "declared_amount", "status"]
].sort_values("record_id")

display(conflict_report.head(20))

# -----------------------------
# 7. RULE-BASED VALIDATION
# -----------------------------

print("\n" + "=" * 75)
print("STEP 4 | RULE-BASED VALIDATION")
print("=" * 75)

rules = {}

rules["invalid_age"] = (
    df["age"].notna() & ~df["age"].between(18, 100)
)

rules["negative_amount"] = (
    df["transaction_amount"].notna()
    & (df["transaction_amount"] < 0)
)

rules["negative_transaction_count"] = (
    df["transaction_count"].notna()
    & (df["transaction_count"] < 0)
)

rules["invalid_status"] = ~df["status"].isin(statuses)

rules["negative_processing_time"] = (
    df["processing_time_seconds"].notna()
    & (df["processing_time_seconds"] < 0)
)

rules["amount_mismatch"] = (
    df["transaction_amount"].notna()
    & df["declared_amount"].notna()
    & (df["transaction_amount"] - df["declared_amount"]).abs().gt(0.01)
)

# Flag rows containing any rule violation.
rule_flags = pd.DataFrame(rules, index=df.index)
rule_flags = rule_flags.fillna(False).astype(bool)

df["rule_violation_count"] = rule_flags.sum(axis=1)
df["rule_violation"] = df["rule_violation_count"] > 0

rule_report = pd.DataFrame({
    "rule": rule_flags.columns,
    "flagged_rows": rule_flags.sum().values
}).sort_values("flagged_rows", ascending=False)

display(rule_report)

# -----------------------------
# 8. MACHINE LEARNING ANOMALIES
# -----------------------------

print("\n" + "=" * 75)
print("STEP 5 | ISOLATION FOREST ANOMALY DETECTION")
print("=" * 75)

numeric_features = [
    "age",
    "transaction_amount",
    "transaction_count",
    "processing_time_seconds",
    "declared_amount"
]

X = df[numeric_features].replace([np.inf, -np.inf], np.nan)

# Imputation and scaling are fitted within the pipeline.
# This model is for anomaly screening, not a fraud classifier.
model = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    IsolationForest(
        n_estimators=200,
        contamination=0.04,
        random_state=SEED
    )
)

predictions = model.fit_predict(X)

df["ml_anomaly"] = predictions == -1

# Lower decision-function scores indicate more unusual records.
df["anomaly_score"] = -model.decision_function(X)

print("Potential ML anomalies:", int(df["ml_anomaly"].sum()))
print("Anomaly detection method: Isolation Forest")
print("Contamination parameter: 4% (configuration, not measured truth)")

# -----------------------------
# 9. COMBINED INVESTIGATION SCORE
# -----------------------------

print("\n" + "=" * 75)
print("STEP 6 | COMBINED INVESTIGATION FLAGS")
print("=" * 75)

df["missing_data_flag"] = df.isna().any(axis=1)

df["exact_duplicate_flag"] = exact_duplicate_mask
df["conflicting_id_flag"] = conflict_mask

df["investigation_flag"] = (
    df["missing_data_flag"]
    | df["exact_duplicate_flag"]
    | df["conflicting_id_flag"]
    | df["rule_violation"]
    | df["ml_anomaly"]
)

def explain_row(row):
    reasons = []

    if row["missing_data_flag"]:
        reasons.append("Missing values")

    if row["exact_duplicate_flag"]:
        reasons.append("Exact duplicate group")

    if row["conflicting_id_flag"]:
        reasons.append("Conflicting record ID")

    if row["rule_violation"]:
        violated = rule_flags.columns[
            rule_flags.loc[row.name].values
        ].tolist()
        reasons.extend(violated)

    if row["ml_anomaly"]:
        reasons.append("Unusual multivariate pattern")

    return "; ".join(dict.fromkeys(reasons)) if reasons else "No issues detected"

df["investigation_reasons"] = df.apply(explain_row, axis=1)

investigation_report = df.loc[
    df["investigation_flag"]
].sort_values(
    ["rule_violation_count", "anomaly_score"],
    ascending=False
)

display(
    investigation_report[
        [
            "record_id",
            "customer_id",
            "timestamp",
            "transaction_amount",
            "age",
            "investigation_flag",
            "anomaly_score",
            "investigation_reasons"
        ]
    ].head(30)
)

# -----------------------------
# 10. DATA RELIABILITY SCORE
# -----------------------------

print("\n" + "=" * 75)
print("STEP 7 | DATA RELIABILITY ASSESSMENT")
print("=" * 75)

# These weights are illustrative design choices, not a validated
# universal standard. The score is an operational quality indicator.

total_rows = max(len(df), 1)

missing_row_rate = df["missing_data_flag"].mean()
duplicate_row_rate = exact_duplicate_mask.mean()
conflict_row_rate = conflict_mask.mean()
rule_violation_rate = df["rule_violation"].mean()

# Count rows with any explicitly measurable issue, without double
# counting overlaps between categories.
explicit_issue_rate = (
    df["missing_data_flag"]
    | exact_duplicate_mask
    | conflict_mask
    | df["rule_violation"]
).mean()

# Penalize explicit issues, not every ML anomaly:
# unsupervised anomalies are candidates for review, not confirmed errors.
reliability_score = max(
    0,
    min(100, 100 * (1 - explicit_issue_rate))
)

if reliability_score >= 95:
    reliability_label = "HIGH DATA COMPLETENESS / RULE COMPLIANCE"
elif reliability_score >= 85:
    reliability_label = "MODERATE DATA QUALITY"
elif reliability_score >= 70:
    reliability_label = "NEEDS INVESTIGATION"
else:
    reliability_label = "LOW QUALITY — REVIEW REQUIRED"

quality_metrics = pd.DataFrame({
    "Metric": [
        "Missing-value row rate (%)",
        "Exact-duplicate row rate (%)",
        "Conflicting-ID row rate (%)",
        "Rule-violation row rate (%)",
        "Rows with one or more explicit issues (%)",
        "Operational reliability score (0–100)",
        "Potential ML anomaly count"
    ],
    "Value": [
        round(missing_row_rate * 100, 2),
        round(duplicate_row_rate * 100, 2),
        round(conflict_row_rate * 100, 2),
        round(rule_violation_rate * 100, 2),
        round(explicit_issue_rate * 100, 2),
        round(reliability_score, 2),
        int(df["ml_anomaly"].sum())
    ]
})

display(quality_metrics)
print("Assessment:", reliability_label)
print(
    "\nInterpretation: This custom score reflects only the explicit "
    "checks implemented here. It is not a certified measure of "
    "overall data truth or business reliability."
)

# -----------------------------
# 11. DATA DRIFT DETECTION
# -----------------------------

print("\n" + "=" * 75)
print("STEP 8 | DATA DRIFT DETECTION")
print("=" * 75)

# Compare earlier records with the more recent 25%.
split = int(len(df) * 0.75)

reference = df.iloc[:split].copy()
current = df.iloc[split:].copy()

drift_rows = []

for col in numeric_features:
    ref = reference[col].dropna()
    cur = current[col].dropna()

    if len(ref) == 0 or len(cur) == 0:
        continue

    ref_mean = ref.mean()
    cur_mean = cur.mean()
    ref_std = ref.std()

    standardized_shift = (
        abs(cur_mean - ref_mean) / ref_std
        if pd.notna(ref_std) and ref_std > 0
        else np.nan
    )

    drift_rows.append({
        "feature": col,
        "reference_mean": round(ref_mean, 3),
        "recent_mean": round(cur_mean, 3),
        "absolute_mean_change": round(abs(cur_mean - ref_mean), 3),
        "standardized_mean_shift": (
            round(standardized_shift, 3)
            if pd.notna(standardized_shift)
            else np.nan
        ),
        "screening_flag": (
            standardized_shift > 0.5
            if pd.notna(standardized_shift)
            else False
        )
    })

drift_report = pd.DataFrame(drift_rows).sort_values(
    "standardized_mean_shift",
    ascending=False
)

display(drift_report)

print(
    "Note: The standardized mean-shift threshold is a simple screening "
    "rule. It is not a statistical significance test and does not "
    "prove that a data pipeline has failed."
)

# -----------------------------
# 12. INTERACTIVE VISUALIZATIONS
# -----------------------------

print("\n" + "=" * 75)
print("STEP 9 | INTERACTIVE VISUALIZATION DASHBOARD")
print("=" * 75)

# Visualization 1: Dataset quality metrics
fig = go.Figure()

fig.add_trace(go.Indicator(
    mode="gauge+number",
    value=reliability_score,
    title={"text": "Operational Data Quality Score"},
    gauge={
        "axis": {"range": [0, 100]},
        "threshold": {
            "line": {"width": 4},
            "thickness": 0.8,
            "value": 85
        }
    }
))

fig.update_layout(
    title="ECHOLOCK | Data Quality Overview",
    height=400
)
fig.show()

# Visualization 2: Missing values by column
missing_plot = missing_report[
    missing_report["missing_count"] > 0
].sort_values("missing_count")

if not missing_plot.empty:
    fig = px.bar(
        missing_plot,
        x="missing_count",
        y="column",
        orientation="h",
        text="missing_count",
        title="Missing Values by Column",
        labels={
            "missing_count": "Missing cells",
            "column": "Feature"
        }
    )
    fig.update_layout(height=450)
    fig.show()

# Visualization 3: Rule violations
fig = px.bar(
    rule_report.sort_values("flagged_rows"),
    x="flagged_rows",
    y="rule",
    orientation="h",
    text="flagged_rows",
    title="Data Validation: Violations by Rule",
    labels={
        "flagged_rows": "Flagged rows",
        "rule": "Validation rule"
    }
)
fig.update_layout(height=500)
fig.show()

# Visualization 4: Anomaly-score distribution
fig = px.histogram(
    df,
    x="anomaly_score",
    color="ml_anomaly",
    nbins=45,
    marginal="box",
    title="Isolation Forest: Anomaly Score Distribution",
    labels={
        "anomaly_score": "Anomaly score (higher = more unusual)",
        "ml_anomaly": "Flagged by model"
    }
)
fig.show()

# Visualization 5: Transaction amount versus processing time
plot_df = df[
    ["transaction_amount", "processing_time_seconds",
     "ml_anomaly", "department"]
].dropna().copy()

# Sample for rendering performance on large datasets.
if len(plot_df) > 1500:
    plot_df = plot_df.sample(1500, random_state=SEED)

fig = px.scatter(
    plot_df,
    x="transaction_amount",
    y="processing_time_seconds",
    color="ml_anomaly",
    symbol="department",
    hover_data=["department"],
    title="Multivariate Anomaly Investigation",
    labels={
        "transaction_amount": "Transaction amount",
        "processing_time_seconds": "Processing time (seconds)",
        "ml_anomaly": "Potential anomaly"
    }
)
fig.show()

# Visualization 6: Reference versus recent mean values
if not drift_report.empty:
    drift_long = drift_report.melt(
        id_vars="feature",
        value_vars=["reference_mean", "recent_mean"],
        var_name="period",
        value_name="mean_value"
    )

    fig = px.bar(
        drift_long,
        x="feature",
        y="mean_value",
        color="period",
        barmode="group",
        title="Data Drift Screening: Reference vs Recent Period",
        labels={
            "feature": "Feature",
            "mean_value": "Mean value",
            "period": "Period"
        }
    )
    fig.update_layout(xaxis_tickangle=-30)
    fig.show()

# Visualization 7: Correlation heatmap
corr = df[numeric_features].corr()

fig = px.imshow(
    corr,
    text_auto=".2f",
    aspect="auto",
    title="Numeric Feature Correlation Matrix",
    labels={"color": "Correlation"}
)
fig.show()

# Visualization 8: Investigation reasons
reason_counts = (
    investigation_report["investigation_reasons"]
    .str.split("; ")
    .explode()
    .value_counts()
    .reset_index()
)
reason_counts.columns = ["reason", "count"]

fig = px.bar(
    reason_counts.head(15).sort_values("count"),
    x="count",
    y="reason",
    orientation="h",
    text="count",
    title="Most Common Investigation Reasons",
    labels={"count": "Occurrences", "reason": "Reason"}
)
fig.update_layout(height=550)
fig.show()

# Visualization 9: PCA projection of numeric records
# PCA is used for visual exploration only, not as proof of anomalies.
pca_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    PCA(n_components=2, random_state=SEED)
)

pca_values = pca_pipeline.fit_transform(df[numeric_features])

pca_df = pd.DataFrame({
    "PC1": pca_values[:, 0],
    "PC2": pca_values[:, 1],
    "Potential anomaly": df["ml_anomaly"].map({
        True: "Flagged",
        False: "Not flagged"
    }).values
})

if len(pca_df) > 2000:
    pca_df = pca_df.sample(2000, random_state=SEED)

fig = px.scatter(
    pca_df,
    x="PC1",
    y="PC2",
    color="Potential anomaly",
    title="PCA Projection of Records",
    labels={
        "PC1": "Principal component 1",
        "PC2": "Principal component 2"
    }
)
fig.show()

# -----------------------------
# 13. EXPORT INVESTIGATION DATA
# -----------------------------

print("\n" + "=" * 75)
print("STEP 10 | EXPORT REPORTS")
print("=" * 75)

# Full dataset with flags
df.to_csv(
    os.path.join(OUTPUT_DIR, "echolock_full_analysis.csv"),
    index=False
)

# Rows requiring review
investigation_report.to_csv(
    os.path.join(OUTPUT_DIR, "echolock_investigation_report.csv"),
    index=False
)

# Data quality report
missing_report.to_csv(
    os.path.join(OUTPUT_DIR, "missing_values_report.csv"),
    index=False
)

rule_report.to_csv(
    os.path.join(OUTPUT_DIR, "validation_rules_report.csv"),
    index=False
)

conflict_report.to_csv(
    os.path.join(OUTPUT_DIR, "conflicting_records_report.csv"),
    index=False
)

drift_report.to_csv(
    os.path.join(OUTPUT_DIR, "data_drift_report.csv"),
    index=False
)

# Export machine-readable summary
summary = {
    "project": "ECHOLOCK",
    "generated_at": datetime.now().isoformat(timespec="seconds"),
    "dataset_type": "Synthetic demonstration data",
    "total_rows": int(len(df)),
    "total_columns": int(len(df.columns)),
    "exact_duplicate_rows": int(exact_duplicate_mask.sum()),
    "unique_conflicting_record_ids": int(len(conflicting_ids)),
    "rows_with_missing_values": int(df["missing_data_flag"].sum()),
    "rows_with_rule_violations": int(df["rule_violation"].sum()),
    "potential_ml_anomalies": int(df["ml_anomaly"].sum()),
    "rows_flagged_for_investigation": int(df["investigation_flag"].sum()),
    "operational_reliability_score": round(float(reliability_score), 2),
    "reliability_interpretation": reliability_label,
    "limitations": [
        "Data is synthetic and results are not real-world findings.",
        "Isolation Forest flags unusual patterns, not confirmed fraud.",
        "Drift screening uses a simple standardized mean-shift threshold.",
        "The reliability score is a custom operational metric.",
        "Flagged records require human review before correction or deletion."
    ]
}

with open(
    os.path.join(OUTPUT_DIR, "echolock_summary.json"),
    "w",
    encoding="utf-8"
) as f:
    json.dump(summary, f, indent=4)

print("Files created successfully:\n")

for filename in sorted(os.listdir(OUTPUT_DIR)):
    filepath = os.path.join(OUTPUT_DIR, filename)
    if os.path.isfile(filepath):
        print(f"  - {filepath} ({os.path.getsize(filepath):,} bytes)")

print("\n" + "=" * 75)
print("ECHOLOCK ANALYSIS COMPLETED")
print("=" * 75)

print(f"""
FINAL SUMMARY
-------------
Total records:                  {len(df):,}
Exact duplicate rows:           {exact_duplicate_mask.sum():,}
Conflicting record IDs:         {len(conflicting_ids):,}
Rows with missing data:          {df['missing_data_flag'].sum():,}
Rows with validation failures:   {df['rule_violation'].sum():,}
Potential ML anomalies:          {df['ml_anomaly'].sum():,}
Rows flagged for investigation:  {df['investigation_flag'].sum():,}
Operational quality score:       {reliability_score:.2f}/100
Assessment:                      {reliability_label}

Output folder: {OUTPUT_DIR}/

IMPORTANT:
Review flagged records before changing or deleting data.
An unusual record is not automatically an incorrect record.
""")

 ECHOLOCK | AI DATA INTEGRITY INTELLIGENCE ENGINE
Generating demonstration data and initializing analysis...

Dataset generated: 1,524 rows
Columns: 11
Output directory: echolock_outputs/

STEP 1 | DATASET OVERVIEW


,Metric,Value
0,Total records,1524
1,Total columns,11
2,Exact duplicate rows,12
3,Rows containing missing values,73
4,Total missing cells,75
5,Unique record IDs,1500
6,Unique customers,297
7,Date range start,2026-08-22 13:00:00
8,Date range end,2026-10-24 00:00:00



STEP 2 | MISSING-VALUE ANALYSIS


,column,missing_count,missing_percentage
6,transaction_amount,35,2.30
5,age,22,1.44
4,region,18,1.18



STEP 3 | DUPLICATE AND CONFLICT ANALYSIS
Rows participating in exact duplicate groups: 24
Unique record IDs with conflicting values: 12
Rows associated with conflicting IDs: 24


...

,Metric,Value
0,Missing-value row rate (%),4.79
1,Exact-duplicate row rate (%),1.57
2,Conflicting-ID row rate (%),1.57
3,Rule-violation row rate (%),3.15
4,Rows with one or more explicit issues (%),10.17
5,Operational reliability score (0–100),89.83
6,Potential ML anomaly count,61.00


Assessment: MODERATE DATA QUALITY

Interpretation: This custom score reflects only the explicit checks implemented here. It is not a certified measure of overall data truth or business reliability.

STEP 8 | DATA DRIFT DETECTION


,feature,reference_mean,recent_mean,absolute_mean_change,standardized_mean_shift,screening_flag
1,transaction_amount,192.485,359.148,166.663,0.980,True
4,declared_amount,196.239,345.862,149.623,0.925,True
3,processing_time_seconds,5.976,8.117,2.141,0.617,True
2,transaction_count,2.969,3.060,0.092,0.052,False
0,age,38.566,38.112,0.453,0.033,False


Note: The standardized mean-shift threshold is a simple screening rule. It is not a statistical significance test and does not prove that a data pipeline has failed.

STEP 9 | INTERACTIVE VISUALIZATION DASHBOARD



STEP 10 | EXPORT REPORTS
Files created successfully:

  - echolock_outputs/conflicting_records_report.csv (1,678 bytes)
  - echolock_outputs/data_drift_report.csv (335 bytes)
  - echolock_outputs/echolock_full_analysis.csv (262,796 bytes)
  - echolock_outputs/echolock_investigation_report.csv (36,295 bytes)
  - echolock_outputs/echolock_summary.json (889 bytes)
  - echolock_outputs/missing_values_report.csv (249 bytes)
  - echolock_outputs/synthetic_raw_dataset.csv (143,410 bytes)
  - echolock_outputs/validation_rules_report.csv (144 bytes)

ECHOLOCK ANALYSIS COMPLETED

FINAL SUMMARY
-------------
Total records:                  1,524
Exact duplicate rows:           24
Conflicting record IDs:         12
Rows with missing data:          73
Rows with validation failures:   48
Potential ML anomalies:          61
Rows flagged for investigation:  205
Operational quality score:       89.83/100
Assessment:                      MODERATE DATA QUALITY

Output folder: echolock_outputs/

IMPORTAN